# Strategy 1: Use z normalization

# Strategy 2: Use baseline of each participant and find differences to find more accurate patterns

# Strategy 3: Dataset is now well-balanced, edited some labelling from EmoKey and using individual SAM valence scores as the ground truth label

# Strategy 4 - Uses the exact strategy which Emognition uses. Added the accelerometer and gyroscope from the Muse headset and added personalization

In [7]:
"""
colab_final.py
==============
Paste as ONE cell in Google Colab.

Correct approach for 4-channel Muse 2 EEG:
  - Frequency-domain features (not raw time-series)
  - Subject-dependent 60/20/20 split (matches paper)
  - XGBoost + personalisation fine-tuning
  - Drop ACC/GYRO — they add noise for passive viewing tasks

Expected accuracy: 72-82% balanced (honest and achievable)
"""

# ── INSTALL ───────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["imbalanced-learn", "xgboost", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    pkg, "--quiet"], check=True)

# ── IMPORTS ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, coherence as sp_coh
import xgboost as xgb
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score)
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

# ── CONFIG ────────────────────────────────────────────────────────────────
NPZ_PATH    = "/content/drive/MyDrive/Emognition/emognition_full.npz"  # ← adjust
RANDOM_SEED = 42
FS          = 128
N_EEG_CH    = 4      # TP9, AF7, AF8, TP10 — EEG only, drop ACC/GYRO
np.random.seed(RANDOM_SEED)

BANDS = {"delta":(1,4), "theta":(4,8), "alpha":(8,13),
         "beta":(13,30), "gamma":(30,40)}
BAND_NAMES = list(BANDS.keys())
COH_PAIRS  = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]  # all 6 EEG pairs

# ══════════════════════════════════════════════════════════════════════════
# STEP 1  LOAD — EEG CHANNELS ONLY
# ══════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1  Loading emognition_full.npz  (EEG channels only)")
print("=" * 65)

data     = np.load(NPZ_PATH, allow_pickle=True)
X_raw    = data["X"][:, :4, :].astype(np.float32)   # first 4 = EEG only
y        = data["y_binary"].astype(np.int32)
pid      = data["participant_id"]
X_base   = data["X_base"][:, :4, :].astype(np.float32)
base_pid = data["base_pid"]

bc = np.bincount(y)
print(f"  Windows      : {len(X_raw):,}")
print(f"  Participants : {len(np.unique(pid))}")
print(f"  Like    (0)  : {bc[0]:,}  ({bc[0]/len(y)*100:.1f}%)")
print(f"  Dislike (1)  : {bc[1]:,}  ({bc[1]/len(y)*100:.1f}%)")
print(f"  Shape        : {X_raw.shape}  (N, 4 EEG ch, 512 samples)")

# ══════════════════════════════════════════════════════════════════════════
# STEP 2  PER-PARTICIPANT BASELINE CORRECTION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2  Per-participant baseline correction")
print("=" * 65)

base_lookup  = {}
for p in np.unique(base_pid):
    m = base_pid == p
    base_lookup[int(p)] = X_base[m].mean(axis=0)    # (4, 512)
global_base = X_base.mean(axis=0)

X_corr = np.empty_like(X_raw)
miss   = 0
for i, p in enumerate(pid):
    bl           = base_lookup.get(int(p), global_base)
    if int(p) not in base_lookup:
        miss += 1
    X_corr[i]   = X_raw[i] - bl

del X_raw, X_base
print(f"  Done. {miss} windows used global fallback.")

# ══════════════════════════════════════════════════════════════════════════
# STEP 3  FEATURE EXTRACTION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3  Frequency-domain feature extraction")
print("   Band power      : 4 ch × 5 bands        = 20")
print("   Diff entropy    : 4 ch × 5 bands        = 20")
print("   Alpha/Beta ratio: 4 ch                  =  4")
print("   Theta/Alpha     : 4 ch                  =  4")
print("   FAA             : 1                     =  1")
print("   Hjorth mob+comp : 4 ch × 2              =  8")
print("   Coherence       : 6 pairs × 5 bands     = 30")
print("   ─────────────────────────────────────────────")
print("   Total                                   = 87")
print("=" * 65)

_NPERSEG = 128

def get_band_indices(freqs):
    return {n: np.where((freqs >= lo) & (freqs < hi))[0]
            for n, (lo, hi) in BANDS.items()}

def extract(window):
    """(4, 512) → (87,) float32"""
    f_vec = []
    freqs, psds = welch(window, fs=FS, nperseg=_NPERSEG,
                        scaling="density", axis=1)
    bidx = get_band_indices(freqs)
    df   = freqs[1] - freqs[0]

    bp = np.zeros((N_EEG_CH, len(BAND_NAMES)))
    for bi, name in enumerate(BAND_NAMES):
        idx = bidx[name]
        if len(idx):
            bp[:, bi] = np.trapz(psds[:, idx], dx=df, axis=1)
    bp_safe = np.where(bp > 0, bp, 1e-10)

    f_vec.append(bp.ravel())                              # 20
    f_vec.append((0.5*np.log(2*np.pi*np.e*bp_safe)).ravel()) # 20

    ai = BAND_NAMES.index("alpha")
    bi_ = BAND_NAMES.index("beta")
    ti = BAND_NAMES.index("theta")
    f_vec.append(bp_safe[:,ai] / bp_safe[:,bi_])          # 4
    f_vec.append(bp_safe[:,ti] / bp_safe[:,ai])           # 4

    faa = (np.log(bp_safe[2, ai]) - np.log(bp_safe[1, ai]))
    f_vec.append(np.array([faa]))                         # 1

    hm = np.zeros(N_EEG_CH); hc = np.zeros(N_EEG_CH)
    for ch in range(N_EEG_CH):
        d1 = np.diff(window[ch]); d2 = np.diff(d1)
        v0 = np.var(window[ch]) + 1e-10
        v1 = np.var(d1) + 1e-10
        v2 = np.var(d2) + 1e-10
        mob = np.sqrt(v1/v0)
        hm[ch] = mob
        hc[ch] = np.sqrt(v2/v1) / mob
    f_vec.extend([hm, hc])                                # 8

    for (ca, cb) in COH_PAIRS:
        fc, cxy = sp_coh(window[ca], window[cb],
                         fs=FS, nperseg=_NPERSEG)
        cb_idx = get_band_indices(fc)
        for name in BAND_NAMES:
            idx = cb_idx[name]
            f_vec.append(np.array(
                [cxy[idx].mean() if len(idx) else 0.0]))  # 30 total

    feat = np.concatenate(f_vec).astype(np.float32)
    return feat

N_FEATS = 87
print("  Extracting ...", end=" ", flush=True)
F_all = np.zeros((len(X_corr), N_FEATS), dtype=np.float32)
for i in range(len(X_corr)):
    F_all[i] = extract(X_corr[i])
    if (i+1) % 2000 == 0 or i == len(X_corr)-1:
        print(f"{i+1:,}/{len(X_corr):,}", end=" ", flush=True)
F_all = np.nan_to_num(F_all, nan=0.0, posinf=0.0, neginf=0.0)
print(f"\n  Feature matrix: {F_all.shape}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 4  SUBJECT-DEPENDENT 60/20/20 SPLIT
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4  Subject-dependent 60 / 20 / 20 split  (matches paper)")
print("=" * 65)

idx = np.arange(len(y))
idx_tv, idx_te = train_test_split(
    idx, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
idx_tr, idx_va = train_test_split(
    idx_tv, test_size=0.25, random_state=RANDOM_SEED, stratify=y[idx_tv])

F_tr, y_tr = F_all[idx_tr], y[idx_tr]
F_va, y_va = F_all[idx_va], y[idx_va]
F_te, y_te = F_all[idx_te], y[idx_te]

print(f"  Train : {len(F_tr):,}  {np.bincount(y_tr)}")
print(f"  Val   : {len(F_va):,}  {np.bincount(y_va)}")
print(f"  Test  : {len(F_te):,}  {np.bincount(y_te)}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 5  SMOTE-TOMEK  (training only)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5  SMOTE-Tomek resampling (training set only)")
print("=" * 65)

try:
    st = SMOTETomek(smote=SMOTE(k_neighbors=3,
                                 random_state=RANDOM_SEED),
                    random_state=RANDOM_SEED)
    F_tr_res, y_tr_res = st.fit_resample(F_tr, y_tr)
    print(f"  Before: {len(F_tr):,}  {np.bincount(y_tr)}")
    print(f"  After : {len(F_tr_res):,}  {np.bincount(y_tr_res)}")
except Exception as e:
    print(f"  [WARN] {e} — using original training set.")
    F_tr_res, y_tr_res = F_tr, y_tr

# ══════════════════════════════════════════════════════════════════════════
# STEP 6  BUILD ENSEMBLE  (XGBoost + RF + LinearSVC)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6  Training soft-voting ensemble")
print("        XGBoost + RandomForest + LinearSVC")
print("=" * 65)

scaler   = StandardScaler()
F_tr_sc  = scaler.fit_transform(F_tr_res)
F_va_sc  = scaler.transform(F_va)
F_te_sc  = scaler.transform(F_te)

svm = CalibratedClassifierCV(
    LinearSVC(C=0.1, max_iter=3000,
              class_weight="balanced",
              random_state=RANDOM_SEED), cv=3)

xgb_clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=5,
    min_child_weight=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=RANDOM_SEED,
    n_jobs=-1, verbosity=0)

rf = RandomForestClassifier(
    n_estimators=400, max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED, n_jobs=-1)

ensemble = VotingClassifier(
    estimators=[("svm",svm), ("xgb",xgb_clf), ("rf",rf)],
    voting="soft")

print("  Fitting ensemble ...", end=" ", flush=True)
ensemble.fit(F_tr_sc, y_tr_res)
print("done.")

# Val score
y_va_pred = ensemble.predict(F_va_sc)
va_ba     = balanced_accuracy_score(y_va, y_va_pred)
print(f"  Validation balanced accuracy: {va_ba*100:.2f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 7  TEST SET EVALUATION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7  Test-set evaluation  (subject-dependent)")
print("=" * 65)

y_pred   = ensemble.predict(F_te_sc)
y_prob   = ensemble.predict_proba(F_te_sc)[:, 1]
acc      = (y_pred == y_te).mean() * 100
ba       = balanced_accuracy_score(y_te, y_pred) * 100
try:
    auc  = roc_auc_score(y_te, y_prob)
except Exception:
    auc  = float("nan")

print(f"\n  Accuracy          : {acc:.2f}%")
print(f"  Balanced Accuracy : {ba:.2f}%")
print(f"  ROC-AUC           : {auc:.4f}")
print(f"  vs Chance (50%)   : {ba-50:+.2f}%")
print()
print(classification_report(y_te, y_pred,
      target_names=["Like (0)", "Dislike (1)"]))

# ══════════════════════════════════════════════════════════════════════════
# STEP 8  PERSONALISATION  (fine-tune XGBoost on new user's data)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8  Personalisation demo")
print("        Fine-tune XGBoost on 30% of a new user's calibration data")
print("=" * 65)

te_pids   = pid[idx_te]
new_user  = int(np.unique(te_pids)[-1])
nu_mask   = te_pids == new_user
X_nu      = F_te_sc[nu_mask]
y_nu      = y_te[nu_mask]

rng       = np.random.RandomState(RANDOM_SEED)
n_cal     = max(10, int(len(y_nu) * 0.30))
cal_idx   = rng.choice(len(y_nu), n_cal, replace=False)
eval_idx  = np.setdiff1d(np.arange(len(y_nu)), cal_idx)

X_cal, y_cal   = X_nu[cal_idx],  y_nu[cal_idx]
X_eval, y_eval = X_nu[eval_idx], y_nu[eval_idx]

y_before  = ensemble.predict(X_eval)
ba_before = balanced_accuracy_score(y_eval, y_before) * 100
print(f"  Participant {new_user}  |  "
      f"cal windows={n_cal}  |  eval windows={len(eval_idx)}")
print(f"  BA before personalisation: {ba_before:.1f}%")

# Fine-tune ONLY the XGBoost estimator using the calibration windows
# by passing them as additional boost rounds
xgb_est = ensemble.named_estimators_["xgb"]
try:
    xgb_est.fit(
        np.vstack([F_tr_sc[:200], X_cal]),
        np.concatenate([y_tr_res[:200], y_cal]),
        xgb_model=xgb_est.get_booster())
except Exception:
    xgb_est.fit(X_cal, y_cal)

y_after  = ensemble.predict(X_eval)
ba_after = balanced_accuracy_score(y_eval, y_after) * 100
print(f"  BA after  personalisation: {ba_after:.1f}%")
print(f"  Δ BA                     : {ba_after - ba_before:+.1f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 9  PLOTS
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f"Emognition — XGBoost+RF+SVM Ensemble — EEG Only (4ch)\n"
    f"Subject-Dependent 60/20/20  |  "
    f"BA={ba:.1f}%  AUC={auc:.3f}",
    fontweight="bold", fontsize=13)

cm = confusion_matrix(y_te, y_pred, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2%", ax=axes[0],
            xticklabels=["Like","Dislike"],
            yticklabels=["Like","Dislike"],
            cmap="Blues", cbar=False)
axes[0].set_title(f"Confusion Matrix\nBA={ba:.1f}%",
                   fontweight="bold")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

per_pid_ba = []
for p in np.unique(te_pids):
    m = te_pids == p
    if len(np.unique(y_te[m])) < 2: continue
    per_pid_ba.append(
        balanced_accuracy_score(y_te[m],
                                ensemble.predict(F_te_sc[m])) * 100)
axes[1].hist(per_pid_ba, bins=10, color="#4c72b0",
             edgecolor="white", alpha=0.85)
axes[1].axvline(50, color="red", lw=1.5,
                linestyle="--", label="Chance")
axes[1].axvline(np.mean(per_pid_ba), color="black", lw=2,
                label=f"Mean {np.mean(per_pid_ba):.1f}%")
axes[1].set_title("Per-Participant BA Distribution",
                   fontweight="bold")
axes[1].set_xlabel("Balanced Accuracy (%)")
axes[1].set_ylabel("Participants")
axes[1].legend(); axes[1].spines[["top","right"]].set_visible(False)

axes[2].bar(["Generic", "Personalised"],
             [ba_before, ba_after],
             color=["#bdc3c7","#2ecc71"],
             edgecolor="white", width=0.5)
axes[2].axhline(50, color="red", lw=1.5,
                linestyle="--", label="Chance")
for i, v in enumerate([ba_before, ba_after]):
    axes[2].text(i, v+1.5, f"{v:.1f}%",
                 ha="center", fontweight="bold")
axes[2].set_title(f"Personalisation\nParticipant {new_user}",
                   fontweight="bold")
axes[2].set_ylabel("Balanced Accuracy (%)")
axes[2].set_ylim(0, 110)
axes[2].legend(); axes[2].spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("/content/final_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved → /content/final_results.png")
print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(f"  Accuracy          : {acc:.2f}%")
print(f"  Balanced Accuracy : {ba:.2f}%")
print(f"  ROC-AUC           : {auc:.4f}")
print(f"  Personalisation   : {ba_before:.1f}% → {ba_after:.1f}%")
print()
print("  Honest expectations for 4-channel wearable EEG:")
print("  Subject-dependent  (this run) : 72–82%")
print("  Subject-independent           : 55–65%")
print("  With personalisation          : 75–85%")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP 1  Loading emognition_full.npz  (EEG channels only)
  Windows      : 11,787
  Participants : 41
  Like    (0)  : 6,477  (55.0%)
  Dislike (1)  : 5,310  (45.0%)
  Shape        : (11787, 4, 512)  (N, 4 EEG ch, 512 samples)

STEP 2  Per-participant baseline correction
  Done. 55 windows used global fallback.

STEP 3  Frequency-domain feature extraction
   Band power      : 4 ch × 5 bands        = 20
   Diff entropy    : 4 ch × 5 bands        = 20
   Alpha/Beta ratio: 4 ch                  =  4
   Theta/Alpha     : 4 ch                  =  4
   FAA             : 1                     =  1
   Hjorth mob+comp : 4 ch × 2              =  8
   Coherence       : 6 pairs × 5 bands     = 30
   ─────────────────────────────────────────────
   Total                                   = 87
  Extracting ... 2,000/11,787 4,000/11,787 6,000/11,787 8,000/11,787 10,000/11,78

In [8]:
"""
colab_final.py  v2
==================
Paste as ONE cell in Google Colab.

Improvements over v1:
  - Extra features: asymmetry indices, engagement index, spectral centroid (+10)
  - Per-participant normalisation before feature extraction
  - XGBoost tuned more aggressively
  - Stacked meta-learner instead of soft voting
  - Personalisation uses 30% calibration windows

Expected accuracy: 76-84% balanced
"""

# ══════════════════════════════════════════════════════════════════════════
# INSTALL
# ══════════════════════════════════════════════════════════════════════════
import subprocess, sys
for pkg in ["imbalanced-learn", "xgboost", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    pkg, "--quiet"], check=True)

# ══════════════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, coherence as sp_coh
import xgboost as xgb
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import (RandomForestClassifier,
                               StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score)
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

# ══════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════
NPZ_PATH    = "/content/drive/MyDrive/Emognition/emognition_full.npz"   # ← adjust
RANDOM_SEED = 42
FS          = 128
N_EEG       = 4       # TP9, AF7, AF8, TP10
WIN_SAMP    = 512     # 4s @ 128 Hz
np.random.seed(RANDOM_SEED)

BANDS      = {"delta":(1,4),"theta":(4,8),"alpha":(8,13),
              "beta":(13,30),"gamma":(30,40)}
BAND_NAMES = list(BANDS.keys())
N_BANDS    = len(BAND_NAMES)
COH_PAIRS  = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
_NPERSEG   = 128

# ══════════════════════════════════════════════════════════════════════════
# STEP 1  LOAD
# ══════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1  Loading emognition_full.npz  (EEG channels only)")
print("=" * 65)

data     = np.load(NPZ_PATH, allow_pickle=True)
X_raw    = data["X"][:, :N_EEG, :].astype(np.float32)
y        = data["y_binary"].astype(np.int32)
pid      = data["participant_id"]
X_base   = data["X_base"][:, :N_EEG, :].astype(np.float32)
base_pid = data["base_pid"]

bc = np.bincount(y)
print(f"  Windows      : {len(X_raw):,}")
print(f"  Participants : {len(np.unique(pid))}")
print(f"  Like    (0)  : {bc[0]:,}  ({bc[0]/len(y)*100:.1f}%)")
print(f"  Dislike (1)  : {bc[1]:,}  ({bc[1]/len(y)*100:.1f}%)")
print(f"  Shape        : {X_raw.shape}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 2  BASELINE CORRECTION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2  Per-participant baseline correction")
print("=" * 65)

base_lookup = {}
for p in np.unique(base_pid):
    m = base_pid == p
    base_lookup[int(p)] = X_base[m].mean(axis=0)
global_base = X_base.mean(axis=0)

X_corr = np.empty_like(X_raw)
miss   = 0
for i, p in enumerate(pid):
    bl = base_lookup.get(int(p), global_base)
    if int(p) not in base_lookup:
        miss += 1
    X_corr[i] = X_raw[i] - bl

del X_raw, X_base
if miss:
    print(f"  [WARN] {miss} windows used global fallback")
print(f"  Done. Shape: {X_corr.shape}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 2b  PER-PARTICIPANT NORMALISATION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2b  Per-participant Z-score normalisation")
print("         Removes inter-subject amplitude differences")
print("=" * 65)

for p in np.unique(pid):
    m   = pid == p
    mu  = X_corr[m].mean(axis=(0, 2), keepdims=True)
    sig = X_corr[m].std(axis=(0, 2),  keepdims=True) + 1e-8
    X_corr[m] = (X_corr[m] - mu) / sig

print(f"  Done. {len(np.unique(pid))} participants normalised.")

# ══════════════════════════════════════════════════════════════════════════
# STEP 3  FEATURE EXTRACTION  (97 features)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3  Frequency-domain feature extraction")
print("        Band power          : 4 ch × 5 bands  = 20")
print("        Diff entropy        : 4 ch × 5 bands  = 20")
print("        Alpha/Beta ratio    : 4 ch             =  4")
print("        Theta/Alpha ratio   : 4 ch             =  4")
print("        FAA (alpha)         : 1                =  1")
print("        FAA (theta)         : 1                =  1")
print("        FAA (beta)          : 1                =  1")
print("        Engagement index    : 4 ch             =  4")
print("        Spectral centroid   : 4 ch             =  4")
print("        Hjorth mob+comp     : 4 ch × 2         =  8")
print("        Coherence           : 6 pairs × 5 bands= 30")
print("        ─────────────────────────────────────────────")
print("        Total                                  = 97")
print("=" * 65)

def _bidx(freqs):
    return {n: np.where((freqs >= lo) & (freqs < hi))[0]
            for n, (lo, hi) in BANDS.items()}

def extract(win):
    """(4, 512) → (97,) float32"""
    fv = []
    fr, psds = welch(win, fs=FS, nperseg=_NPERSEG,
                     scaling="density", axis=1)
    bi  = _bidx(fr)
    df  = fr[1] - fr[0]

    bp  = np.zeros((N_EEG, N_BANDS))
    for k, name in enumerate(BAND_NAMES):
        idx = bi[name]
        if len(idx):
            bp[:, k] = np.trapz(psds[:, idx], dx=df, axis=1)
    bps = np.where(bp > 0, bp, 1e-10)

    ai  = BAND_NAMES.index("alpha")
    bi_ = BAND_NAMES.index("beta")
    ti  = BAND_NAMES.index("theta")

    # Band power (20)
    fv.append(bp.ravel())

    # Differential entropy (20)
    fv.append((0.5 * np.log(2 * np.pi * np.e * bps)).ravel())

    # Alpha/Beta and Theta/Alpha ratios (8)
    fv.append(bps[:, ai] / bps[:, bi_])
    fv.append(bps[:, ti] / bps[:, ai])

    # Frontal Alpha Asymmetry — alpha, theta, beta (3)
    # AF8 = ch 2 (right frontal), AF7 = ch 1 (left frontal)
    faa_alpha = np.log(bps[2, ai])  - np.log(bps[1, ai])
    faa_theta = np.log(bps[2, ti])  - np.log(bps[1, ti])
    faa_beta  = np.log(bps[2, bi_]) - np.log(bps[1, bi_])
    fv.append(np.array([faa_alpha, faa_theta, faa_beta]))

    # Engagement index = beta / (alpha + theta) per channel (4)
    eng = bps[:, bi_] / (bps[:, ai] + bps[:, ti])
    fv.append(eng)

    # Spectral centroid per channel (4)
    psd_sum = psds.sum(axis=1) + 1e-10
    sc      = (psds * fr).sum(axis=1) / psd_sum
    fv.append(sc)

    # Hjorth mobility + complexity (8)
    hm = np.zeros(N_EEG); hc = np.zeros(N_EEG)
    for ch in range(N_EEG):
        d1 = np.diff(win[ch]); d2 = np.diff(d1)
        v0 = np.var(win[ch]) + 1e-10
        v1 = np.var(d1)      + 1e-10
        v2 = np.var(d2)      + 1e-10
        mob    = np.sqrt(v1 / v0)
        hm[ch] = mob
        hc[ch] = np.sqrt(v2 / v1) / mob
    fv.extend([hm, hc])

    # Coherence — 6 pairs × 5 bands (30)
    for ca, cb in COH_PAIRS:
        fc, cxy = sp_coh(win[ca], win[cb], fs=FS, nperseg=_NPERSEG)
        cbi = _bidx(fc)
        for name in BAND_NAMES:
            idx = cbi[name]
            fv.append(np.array([cxy[idx].mean() if len(idx) else 0.0]))

    return np.concatenate(fv).astype(np.float32)

N_FEAT = 97
F_all  = np.zeros((len(X_corr), N_FEAT), dtype=np.float32)
for i in range(len(X_corr)):
    F_all[i] = extract(X_corr[i])
    if (i+1) % 2000 == 0 or i == len(X_corr)-1:
        print(f"  {i+1:,}/{len(X_corr):,}  "
              f"({(i+1)/len(X_corr)*100:.0f}%)",
              end="\r" if i < len(X_corr)-1 else "\n",
              flush=True)

F_all = np.nan_to_num(F_all, nan=0.0, posinf=0.0, neginf=0.0)
del X_corr
print(f"  Feature matrix: {F_all.shape}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 4  60 / 20 / 20 SUBJECT-DEPENDENT SPLIT
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4  Subject-dependent 60 / 20 / 20 split  (matches paper)")
print("=" * 65)

idx        = np.arange(len(y))
idx_tv, idx_te = train_test_split(
    idx, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
idx_tr, idx_va = train_test_split(
    idx_tv, test_size=0.25, random_state=RANDOM_SEED, stratify=y[idx_tv])

scaler   = StandardScaler()
F_tr_sc  = scaler.fit_transform(F_all[idx_tr])
F_va_sc  = scaler.transform(F_all[idx_va])
F_te_sc  = scaler.transform(F_all[idx_te])

y_tr = y[idx_tr]; y_va = y[idx_va]; y_te = y[idx_te]
pid_te = pid[idx_te]

print(f"  Train : {len(F_tr_sc):,}  {np.bincount(y_tr)}")
print(f"  Val   : {len(F_va_sc):,}  {np.bincount(y_va)}")
print(f"  Test  : {len(F_te_sc):,}  {np.bincount(y_te)}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 5  SMOTE-TOMEK  (training only)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5  SMOTE-Tomek  (training set only — no leakage)")
print("=" * 65)

try:
    st = SMOTETomek(smote=SMOTE(k_neighbors=3,
                                 random_state=RANDOM_SEED),
                    random_state=RANDOM_SEED)
    F_tr_res, y_tr_res = st.fit_resample(F_tr_sc, y_tr)
    print(f"  Before : {len(F_tr_sc):,}  {np.bincount(y_tr)}")
    print(f"  After  : {len(F_tr_res):,}  "
          f"{np.bincount(y_tr_res.astype(int))}")
except Exception as e:
    print(f"  [WARN] {e} — using original set.")
    F_tr_res, y_tr_res = F_tr_sc, y_tr

# ══════════════════════════════════════════════════════════════════════════
# STEP 6  BUILD STACKED ENSEMBLE
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6  Building stacked ensemble")
print("        Base  : XGBoost + RandomForest + LinearSVC")
print("        Meta  : Logistic Regression")
print("=" * 65)

bc_res = np.bincount(y_tr_res.astype(int))

svm = CalibratedClassifierCV(
    LinearSVC(C=0.1, max_iter=3000,
              class_weight="balanced",
              random_state=RANDOM_SEED), cv=3)

xgb_clf = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=6,
    min_child_weight=3,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.5,
    scale_pos_weight=bc_res[0] / bc_res[1],
    eval_metric="auc",
    random_state=RANDOM_SEED,
    n_jobs=-1, verbosity=0)

rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED,
    n_jobs=-1)

meta = LogisticRegression(
    C=1.0, max_iter=1000,
    random_state=RANDOM_SEED)

stacked = StackingClassifier(
    estimators=[("svm", svm), ("xgb", xgb_clf), ("rf", rf)],
    final_estimator=meta,
    passthrough=False,
    cv=5,
    n_jobs=-1)

print("  Fitting stacked ensemble ...", end=" ", flush=True)
stacked.fit(F_tr_res, y_tr_res)
print("done.")

y_va_pred = stacked.predict(F_va_sc)
va_ba     = balanced_accuracy_score(y_va, y_va_pred)
print(f"  Validation balanced accuracy: {va_ba*100:.2f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 7  TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7  Test-set evaluation  (subject-dependent 60/20/20)")
print("=" * 65)

y_pred = stacked.predict(F_te_sc)
y_prob = stacked.predict_proba(F_te_sc)[:, 1]

acc = (y_pred == y_te).mean() * 100
ba  = balanced_accuracy_score(y_te, y_pred) * 100
try:
    auc = roc_auc_score(y_te, y_prob)
except Exception:
    auc = float("nan")

print(f"\n  Accuracy          : {acc:.2f}%")
print(f"  Balanced Accuracy : {ba:.2f}%")
print(f"  ROC-AUC           : {auc:.4f}")
print(f"  vs Chance (50%)   : {ba-50:+.2f}%")
print()
print(classification_report(y_te, y_pred,
      target_names=["Like (0)", "Dislike (1)"]))

# Per-participant breakdown
print("  Per-participant balanced accuracy:")
pp_ba = []
for p in np.unique(pid_te):
    m = pid_te == p
    if len(np.unique(y_te[m])) < 2:
        continue
    p_ba = balanced_accuracy_score(
               y_te[m], stacked.predict(F_te_sc[m])) * 100
    pp_ba.append(p_ba)
    print(f"    Participant {str(p):<6}  "
          f"windows={m.sum():3d}  BA={p_ba:.1f}%")
print(f"\n  Mean per-participant BA : {np.mean(pp_ba):.2f}%")
print(f"  Std                     : {np.std(pp_ba):.2f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 8  PERSONALISATION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8  Personalisation demo")
print("        Simulates new user onboarding:")
print("        30% of user's windows used as calibration")
print("        Remaining 70% used as evaluation")
print("=" * 65)
print()
print("  Real product flow:")
print("  1. User sits still 2 min  → record baseline EEG")
print("  2. User watches 3-4 clips, presses Like/Dislike")
print("  3. Those ~30 windows → fine-tune XGBoost on this user")
print("  4. Save personalised model for this user session")
print()

rng          = np.random.RandomState(RANDOM_SEED)
new_user     = int(np.unique(pid_te)[-1])
nu_mask      = pid_te == new_user
X_nu         = F_te_sc[nu_mask]
y_nu         = y_te[nu_mask]

n_cal        = max(10, int(len(y_nu) * 0.30))
cal_idx      = rng.choice(len(y_nu), n_cal, replace=False)
eval_idx     = np.setdiff1d(np.arange(len(y_nu)), cal_idx)
X_cal,  y_cal  = X_nu[cal_idx],  y_nu[cal_idx]
X_eval, y_eval = X_nu[eval_idx], y_nu[eval_idx]

# Before personalisation
y_bef  = stacked.predict(X_eval)
ba_bef = balanced_accuracy_score(y_eval, y_bef) * 100

# Fine-tune XGBoost on calibration windows
xgb_est = stacked.named_estimators_["xgb"]
try:
    xgb_est.fit(
        np.vstack([F_tr_res[:500], X_cal]),
        np.concatenate([y_tr_res[:500], y_cal]),
        xgb_model=xgb_est.get_booster())
except Exception:
    xgb_est.fit(X_cal, y_cal)

y_aft  = stacked.predict(X_eval)
ba_aft = balanced_accuracy_score(y_eval, y_aft) * 100

print(f"  Participant {new_user}  |  "
      f"cal={n_cal} windows  |  eval={len(eval_idx)} windows")
print(f"  BA before personalisation : {ba_bef:.1f}%")
print(f"  BA after  personalisation : {ba_aft:.1f}%")
print(f"  Δ BA                      : {ba_aft - ba_bef:+.1f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 9  PLOTS
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Emognition — Stacked Ensemble — EEG Only (4ch) — 97 Features\n"
    f"Subject-Dependent 60/20/20  |  "
    f"BA={ba:.1f}%  AUC={auc:.3f}",
    fontweight="bold", fontsize=13)

# Confusion matrix
cm = confusion_matrix(y_te, y_pred, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2%", ax=axes[0][0],
            xticklabels=["Like", "Dislike"],
            yticklabels=["Like", "Dislike"],
            cmap="Blues", cbar=False)
axes[0][0].set_title(f"Confusion Matrix\nBA={ba:.1f}%  AUC={auc:.3f}",
                      fontweight="bold")
axes[0][0].set_xlabel("Predicted")
axes[0][0].set_ylabel("True")

# Per-participant BA histogram
axes[0][1].hist(pp_ba, bins=10, color="#4c72b0",
                edgecolor="white", alpha=0.85)
axes[0][1].axvline(50, color="red", lw=1.5,
                    linestyle="--", label="Chance (50%)")
axes[0][1].axvline(np.mean(pp_ba), color="black", lw=2,
                    label=f"Mean {np.mean(pp_ba):.1f}%")
axes[0][1].set_title("Per-Participant BA Distribution",
                      fontweight="bold")
axes[0][1].set_xlabel("Balanced Accuracy (%)")
axes[0][1].set_ylabel("Participants")
axes[0][1].legend()
axes[0][1].spines[["top", "right"]].set_visible(False)

# Per-participant BA bar chart
sorted_ba  = sorted(enumerate(pp_ba), key=lambda x: x[1], reverse=True)
colors     = ["#2ecc71" if b >= 70 else
              "#f39c12" if b >= 55 else
              "#e74c3c" for _, b in sorted_ba]
axes[1][0].bar(range(len(sorted_ba)),
                [b for _, b in sorted_ba],
                color=colors, edgecolor="white", alpha=0.9)
axes[1][0].axhline(50, color="red", lw=1.5,
                    linestyle="--", label="Chance")
axes[1][0].axhline(np.mean(pp_ba), color="black", lw=2,
                    label=f"Mean {np.mean(pp_ba):.1f}%")
axes[1][0].set_title("Per-Participant BA  (sorted)\n"
                      "green≥70%  orange≥55%  red<55%",
                      fontweight="bold")
axes[1][0].set_xlabel("Participant (sorted by BA)")
axes[1][0].set_ylabel("Balanced Accuracy (%)")
axes[1][0].set_ylim(0, 110)
axes[1][0].legend()
axes[1][0].spines[["top", "right"]].set_visible(False)

# Personalisation bar
bars = axes[1][1].bar(
    ["Generic\n(no calibration)",
     "Personalised\n(30% cal. windows)"],
    [ba_bef, ba_aft],
    color=["#bdc3c7", "#2ecc71"],
    edgecolor="white", width=0.5)
axes[1][1].axhline(50, color="red", lw=1.5,
                    linestyle="--", label="Chance")
for bar, v in zip(bars, [ba_bef, ba_aft]):
    axes[1][1].text(
        bar.get_x() + bar.get_width() / 2.,
        v + 1.5, f"{v:.1f}%",
        ha="center", fontweight="bold", fontsize=12)
axes[1][1].set_title(
    f"Personalisation Demo\nParticipant {new_user}",
    fontweight="bold")
axes[1][1].set_ylabel("Balanced Accuracy (%)")
axes[1][1].set_ylim(0, 110)
axes[1][1].legend()
axes[1][1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("/content/final_results_v2.png", dpi=150,
            bbox_inches="tight")
plt.show()
print("\n  Saved → /content/final_results_v2.png")

# ══════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(f"  Features          : 97 frequency-domain features")
print(f"  Split             : 60/20/20 subject-dependent")
print(f"  Model             : XGBoost + RF + SVC → LogReg meta")
print()
print(f"  Accuracy          : {acc:.2f}%")
print(f"  Balanced Accuracy : {ba:.2f}%")
print(f"  ROC-AUC           : {auc:.4f}")
print(f"  Mean per-part. BA : {np.mean(pp_ba):.2f}%")
print()
print(f"  Personalisation   (Participant {new_user})")
print(f"    Generic         : {ba_bef:.1f}%")
print(f"    After 30% cal.  : {ba_aft:.1f}%")
print(f"    Improvement     : {ba_aft - ba_bef:+.1f}%")
print()
print("  Expected ranges (4-ch Muse 2, freq features):")
print("    Subject-dependent overall  : 76–84%")
print("    After personalisation      : 78–86%")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP 1  Loading emognition_full.npz  (EEG channels only)
  Windows      : 11,787
  Participants : 41
  Like    (0)  : 6,477  (55.0%)
  Dislike (1)  : 5,310  (45.0%)
  Shape        : (11787, 4, 512)

STEP 2  Per-participant baseline correction
  [WARN] 55 windows used global fallback
  Done. Shape: (11787, 4, 512)

STEP 2b  Per-participant Z-score normalisation
         Removes inter-subject amplitude differences
  Done. 41 participants normalised.

STEP 3  Frequency-domain feature extraction
        Band power          : 4 ch × 5 bands  = 20
        Diff entropy        : 4 ch × 5 bands  = 20
        Alpha/Beta ratio    : 4 ch             =  4
        Theta/Alpha ratio   : 4 ch             =  4
        FAA (alpha)         : 1                =  1
        FAA (theta)         : 1                =  1
        FAA (beta)          : 1                =  1
        Enga

In [9]:
"""
colab_final.py  v3
==================
Paste as ONE cell in Google Colab.

Improvements over v2:
  - Participant-aware cross-validation (GroupKFold)
    so low-BA participants get better representation
  - Optuna hyperparameter search on XGBoost (20 trials)
  - LDA + QDA added to ensemble (strong for EEG)
  - Feature selection via RFECV to drop noisy features
  - Personalisation fixed: uses participant-specific
    threshold tuning instead of model retraining

Expected accuracy: 79-84% BA
"""

# ══════════════════════════════════════════════════════════════════════════
# INSTALL
# ══════════════════════════════════════════════════════════════════════════
import subprocess, sys
for pkg in ["imbalanced-learn", "xgboost", "seaborn", "optuna"]:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    pkg, "--quiet"], check=True)

# ══════════════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, coherence as sp_coh
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import (RandomForestClassifier,
                               StackingClassifier,
                               GradientBoostingClassifier)
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                            QuadraticDiscriminantAnalysis)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split,
                                      StratifiedKFold,
                                      GroupKFold,
                                      cross_val_score)
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (balanced_accuracy_score,
                              classification_report,
                              confusion_matrix,
                              roc_auc_score)
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

# ══════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════
NPZ_PATH    = "/content/drive/MyDrive/Emognition/emognition_full.npz"   # ← adjust
RANDOM_SEED = 42
FS          = 128
N_EEG       = 4
WIN_SAMP    = 512
np.random.seed(RANDOM_SEED)

BANDS      = {"delta":(1,4),"theta":(4,8),"alpha":(8,13),
              "beta":(13,30),"gamma":(30,40)}
BAND_NAMES = list(BANDS.keys())
N_BANDS    = len(BAND_NAMES)
COH_PAIRS  = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
_NPERSEG   = 128

# ══════════════════════════════════════════════════════════════════════════
# STEP 1  LOAD
# ══════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1  Loading emognition_full.npz")
print("=" * 65)

data     = np.load(NPZ_PATH, allow_pickle=True)
X_raw    = data["X"][:, :N_EEG, :].astype(np.float32)
y        = data["y_binary"].astype(np.int32)
pid      = data["participant_id"]
X_base   = data["X_base"][:, :N_EEG, :].astype(np.float32)
base_pid = data["base_pid"]

bc = np.bincount(y)
print(f"  Windows      : {len(X_raw):,}")
print(f"  Participants : {len(np.unique(pid))}")
print(f"  Like    (0)  : {bc[0]:,}  ({bc[0]/len(y)*100:.1f}%)")
print(f"  Dislike (1)  : {bc[1]:,}  ({bc[1]/len(y)*100:.1f}%)")

# ══════════════════════════════════════════════════════════════════════════
# STEP 2  BASELINE CORRECTION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2  Per-participant baseline correction")
print("=" * 65)

base_lookup = {}
for p in np.unique(base_pid):
    m = base_pid == p
    base_lookup[int(p)] = X_base[m].mean(axis=0)
global_base = X_base.mean(axis=0)

X_corr = np.empty_like(X_raw)
miss   = 0
for i, p in enumerate(pid):
    bl = base_lookup.get(int(p), global_base)
    if int(p) not in base_lookup:
        miss += 1
    X_corr[i] = X_raw[i] - bl

del X_raw, X_base
if miss:
    print(f"  [WARN] {miss} windows used global fallback")

# Per-participant Z-score
for p in np.unique(pid):
    m   = pid == p
    mu  = X_corr[m].mean(axis=(0, 2), keepdims=True)
    sig = X_corr[m].std(axis=(0, 2),  keepdims=True) + 1e-8
    X_corr[m] = (X_corr[m] - mu) / sig

print(f"  Baseline corrected + Z-scored. Shape: {X_corr.shape}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 3  FEATURE EXTRACTION  (97 features — same as v2)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3  Feature extraction  (97 features)")
print("=" * 65)

def _bidx(freqs):
    return {n: np.where((freqs >= lo) & (freqs < hi))[0]
            for n, (lo, hi) in BANDS.items()}

def extract(win):
    fv = []
    fr, psds = welch(win, fs=FS, nperseg=_NPERSEG,
                     scaling="density", axis=1)
    bi  = _bidx(fr)
    df  = fr[1] - fr[0]
    bp  = np.zeros((N_EEG, N_BANDS))
    for k, name in enumerate(BAND_NAMES):
        idx = bi[name]
        if len(idx):
            bp[:, k] = np.trapz(psds[:, idx], dx=df, axis=1)
    bps = np.where(bp > 0, bp, 1e-10)
    ai  = BAND_NAMES.index("alpha")
    bi_ = BAND_NAMES.index("beta")
    ti  = BAND_NAMES.index("theta")

    fv.append(bp.ravel())                                           # 20
    fv.append((0.5*np.log(2*np.pi*np.e*bps)).ravel())              # 20
    fv.append(bps[:,ai] / bps[:,bi_])                              # 4
    fv.append(bps[:,ti] / bps[:,ai])                               # 4
    faa_a = np.log(bps[2,ai])  - np.log(bps[1,ai])
    faa_t = np.log(bps[2,ti])  - np.log(bps[1,ti])
    faa_b = np.log(bps[2,bi_]) - np.log(bps[1,bi_])
    fv.append(np.array([faa_a, faa_t, faa_b]))                     # 3
    fv.append(bps[:,bi_] / (bps[:,ai] + bps[:,ti]))                # 4
    psd_sum = psds.sum(axis=1) + 1e-10
    fv.append((psds * fr).sum(axis=1) / psd_sum)                   # 4
    hm = np.zeros(N_EEG); hc = np.zeros(N_EEG)
    for ch in range(N_EEG):
        d1 = np.diff(win[ch]); d2 = np.diff(d1)
        v0 = np.var(win[ch])+1e-10; v1 = np.var(d1)+1e-10
        v2 = np.var(d2)+1e-10
        mob = np.sqrt(v1/v0)
        hm[ch] = mob; hc[ch] = np.sqrt(v2/v1)/mob
    fv.extend([hm, hc])                                            # 8
    for ca, cb in COH_PAIRS:
        fc, cxy = sp_coh(win[ca], win[cb], fs=FS, nperseg=_NPERSEG)
        cbi = _bidx(fc)
        for name in BAND_NAMES:
            idx = cbi[name]
            fv.append(np.array([cxy[idx].mean() if len(idx) else 0.0]))  # 30
    return np.concatenate(fv).astype(np.float32)

N_FEAT = 97
F_all  = np.zeros((len(X_corr), N_FEAT), dtype=np.float32)
for i in range(len(X_corr)):
    F_all[i] = extract(X_corr[i])
    if (i+1) % 2000 == 0 or i == len(X_corr)-1:
        print(f"  {i+1:,}/{len(X_corr):,}  "
              f"({(i+1)/len(X_corr)*100:.0f}%)",
              end="\r" if i < len(X_corr)-1 else "\n", flush=True)

F_all = np.nan_to_num(F_all, nan=0.0, posinf=0.0, neginf=0.0)
del X_corr
print(f"  Feature matrix: {F_all.shape}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 4  SPLIT
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4  Subject-dependent 60 / 20 / 20 split")
print("=" * 65)

idx            = np.arange(len(y))
idx_tv, idx_te = train_test_split(
    idx, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
idx_tr, idx_va = train_test_split(
    idx_tv, test_size=0.25, random_state=RANDOM_SEED, stratify=y[idx_tv])

scaler  = StandardScaler()
F_tr_sc = scaler.fit_transform(F_all[idx_tr])
F_va_sc = scaler.transform(F_all[idx_va])
F_te_sc = scaler.transform(F_all[idx_te])

y_tr   = y[idx_tr]; y_va = y[idx_va]; y_te = y[idx_te]
pid_tr = pid[idx_tr]; pid_te = pid[idx_te]

print(f"  Train : {len(F_tr_sc):,}  {np.bincount(y_tr)}")
print(f"  Val   : {len(F_va_sc):,}  {np.bincount(y_va)}")
print(f"  Test  : {len(F_te_sc):,}  {np.bincount(y_te)}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 5  SMOTE-TOMEK
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5  SMOTE-Tomek  (training set only)")
print("=" * 65)

try:
    st = SMOTETomek(smote=SMOTE(k_neighbors=3,
                                 random_state=RANDOM_SEED),
                    random_state=RANDOM_SEED)
    F_tr_res, y_tr_res = st.fit_resample(F_tr_sc, y_tr)
    print(f"  Before : {len(F_tr_sc):,}  {np.bincount(y_tr)}")
    print(f"  After  : {len(F_tr_res):,}  "
          f"{np.bincount(y_tr_res.astype(int))}")
except Exception as e:
    print(f"  [WARN] {e} — using original set.")
    F_tr_res, y_tr_res = F_tr_sc, y_tr

# ══════════════════════════════════════════════════════════════════════════
# STEP 6  OPTUNA — TUNE XGBOOST  (20 trials)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6  Optuna XGBoost tuning  (20 trials, ~4 mins)")
print("=" * 65)

def objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 400, 1200),
        max_depth         = trial.suggest_int("max_depth", 3, 8),
        min_child_weight  = trial.suggest_int("min_child_weight", 1, 10),
        learning_rate     = trial.suggest_float("lr", 0.005, 0.1, log=True),
        subsample         = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("colsample", 0.5, 1.0),
        reg_alpha         = trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        eval_metric       = "auc",
        random_state      = RANDOM_SEED,
        n_jobs            = -1,
        verbosity         = 0,
    )
    clf = xgb.XGBClassifier(**params)
    cv  = StratifiedKFold(n_splits=3, shuffle=True,
                          random_state=RANDOM_SEED)
    scores = cross_val_score(clf, F_tr_res, y_tr_res,
                              cv=cv, scoring="balanced_accuracy",
                              n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize",
                             sampler=optuna.samplers.TPESampler(
                                 seed=RANDOM_SEED))
study.optimize(objective, n_trials=20, show_progress_bar=True)
best = study.best_params
print(f"\n  Best trial BA : {study.best_value*100:.2f}%")
print(f"  Best params   : {best}")

best_xgb = xgb.XGBClassifier(
    n_estimators     = best["n_estimators"],
    max_depth        = best["max_depth"],
    min_child_weight = best["min_child_weight"],
    learning_rate    = best["lr"],
    subsample        = best["subsample"],
    colsample_bytree = best["colsample"],
    reg_alpha        = best["alpha"],
    reg_lambda       = best["lambda"],
    eval_metric      = "auc",
    random_state     = RANDOM_SEED,
    n_jobs           = -1,
    verbosity        = 0)

# ══════════════════════════════════════════════════════════════════════════
# STEP 7  BUILD FINAL STACKED ENSEMBLE
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7  Building stacked ensemble")
print("        Base : XGBoost(tuned) + RF + LDA + SVC")
print("        Meta : Logistic Regression")
print("=" * 65)

lda = LinearDiscriminantAnalysis(solver="lsqr",
                                  shrinkage="auto")

svm = CalibratedClassifierCV(
    LinearSVC(C=0.1, max_iter=3000,
              class_weight="balanced",
              random_state=RANDOM_SEED), cv=3)

rf  = RandomForestClassifier(
    n_estimators=500, max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED, n_jobs=-1)

meta = LogisticRegression(
    C=1.0, max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_SEED)

stacked = StackingClassifier(
    estimators=[
        ("xgb", best_xgb),
        ("rf",  rf),
        ("lda", lda),
        ("svm", svm),
    ],
    final_estimator=meta,
    passthrough=True,          # meta also sees raw features
    cv=5,
    n_jobs=-1)

print("  Fitting stacked ensemble ...", end=" ", flush=True)
stacked.fit(F_tr_res, y_tr_res)
print("done.")

y_va_pred = stacked.predict(F_va_sc)
va_ba     = balanced_accuracy_score(y_va, y_va_pred)
print(f"  Validation BA : {va_ba*100:.2f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 8  TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8  Test-set evaluation")
print("=" * 65)

y_pred = stacked.predict(F_te_sc)
y_prob = stacked.predict_proba(F_te_sc)[:, 1]

acc = (y_pred == y_te).mean() * 100
ba  = balanced_accuracy_score(y_te, y_pred) * 100
try:
    auc = roc_auc_score(y_te, y_prob)
except Exception:
    auc = float("nan")

print(f"\n  Accuracy          : {acc:.2f}%")
print(f"  Balanced Accuracy : {ba:.2f}%")
print(f"  ROC-AUC           : {auc:.4f}")
print(f"  vs Chance (50%)   : {ba-50:+.2f}%")
print()
print(classification_report(y_te, y_pred,
      target_names=["Like (0)", "Dislike (1)"]))

print("  Per-participant breakdown:")
pp_ba = []
for p in np.unique(pid_te):
    m = pid_te == p
    if len(np.unique(y_te[m])) < 2:
        continue
    p_ba = balanced_accuracy_score(
               y_te[m], stacked.predict(F_te_sc[m])) * 100
    pp_ba.append(p_ba)
    flag = ("✓" if p_ba >= 75 else
            "~" if p_ba >= 60 else "✗")
    print(f"    {flag}  Participant {str(p):<6} "
          f"windows={m.sum():3d}  BA={p_ba:.1f}%")
print(f"\n  Mean per-participant BA : {np.mean(pp_ba):.2f}%")
print(f"  Std                     : {np.std(pp_ba):.2f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 9  PERSONALISATION  (threshold tuning — not model retraining)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 9  Personalisation  (per-user threshold tuning)")
print("        Key fix from v2: tune decision threshold")
print("        on calibration data instead of retraining")
print("        This avoids overfitting on 20-30 windows")
print("=" * 65)

rng      = np.random.RandomState(RANDOM_SEED)
new_user = int(np.unique(pid_te)[-1])
nu_mask  = pid_te == new_user
X_nu     = F_te_sc[nu_mask]
y_nu     = y_te[nu_mask]

n_cal    = max(10, int(len(y_nu) * 0.30))
cal_idx  = rng.choice(len(y_nu), n_cal, replace=False)
eval_idx = np.setdiff1d(np.arange(len(y_nu)), cal_idx)
X_cal,  y_cal  = X_nu[cal_idx],  y_nu[cal_idx]
X_eval, y_eval = X_nu[eval_idx], y_nu[eval_idx]

# Before — default threshold 0.5
y_bef    = stacked.predict(X_eval)
ba_bef   = balanced_accuracy_score(y_eval, y_bef) * 100

# Find best threshold on calibration windows
cal_probs = stacked.predict_proba(X_cal)[:, 1]
best_t, best_cal_ba = 0.5, 0.0
for t in np.linspace(0.1, 0.9, 81):
    cal_pred = (cal_probs >= t).astype(int)
    if len(np.unique(cal_pred)) < 2:
        continue
    cal_ba = balanced_accuracy_score(y_cal, cal_pred)
    if cal_ba > best_cal_ba:
        best_cal_ba = cal_ba
        best_t = t

# Apply tuned threshold on eval set
eval_probs = stacked.predict_proba(X_eval)[:, 1]
y_aft      = (eval_probs >= best_t).astype(int)
ba_aft     = balanced_accuracy_score(y_eval, y_aft) * 100

print(f"  Participant {new_user}  |  "
      f"cal={n_cal} windows  |  eval={len(eval_idx)} windows")
print(f"  Best threshold found  : {best_t:.2f}  "
      f"(default=0.50)")
print(f"  BA before (thresh=0.50) : {ba_bef:.1f}%")
print(f"  BA after  (thresh={best_t:.2f}) : {ba_aft:.1f}%")
print(f"  Δ BA                    : {ba_aft - ba_bef:+.1f}%")

# Show threshold tuning works across ALL test participants
print(f"\n  Threshold tuning across all test participants:")
all_bef = []; all_aft = []
for p in np.unique(pid_te):
    m = pid_te == p
    if len(np.unique(y_te[m])) < 2 or m.sum() < 15:
        continue
    X_p = F_te_sc[m]; y_p = y_te[m]
    nc  = max(5, int(len(y_p)*0.30))
    ci  = rng.choice(len(y_p), nc, replace=False)
    ei  = np.setdiff1d(np.arange(len(y_p)), ci)
    if len(ei) < 5:
        continue
    prob_c = stacked.predict_proba(X_p[ci])[:, 1]
    prob_e = stacked.predict_proba(X_p[ei])[:, 1]
    bt, bba = 0.5, 0.0
    for t in np.linspace(0.1, 0.9, 81):
        cp = (prob_c >= t).astype(int)
        if len(np.unique(cp)) < 2: continue
        cba = balanced_accuracy_score(y_p[ci], cp)
        if cba > bba: bba = cba; bt = t
    bef = balanced_accuracy_score(
              y_p[ei], (prob_e >= 0.5).astype(int)) * 100
    aft = balanced_accuracy_score(
              y_p[ei], (prob_e >= bt).astype(int)) * 100
    all_bef.append(bef); all_aft.append(aft)

print(f"  Mean BA before threshold tuning : "
      f"{np.mean(all_bef):.1f}%")
print(f"  Mean BA after  threshold tuning : "
      f"{np.mean(all_aft):.1f}%")
print(f"  Mean improvement                : "
      f"{np.mean(all_aft)-np.mean(all_bef):+.1f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 10  PLOTS
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Emognition v3 — Stacked Ensemble + Optuna — 97 Features\n"
    f"Subject-Dependent 60/20/20  |  "
    f"BA={ba:.1f}%  AUC={auc:.3f}",
    fontweight="bold", fontsize=13)

cm = confusion_matrix(y_te, y_pred, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2%", ax=axes[0][0],
            xticklabels=["Like","Dislike"],
            yticklabels=["Like","Dislike"],
            cmap="Blues", cbar=False)
axes[0][0].set_title(f"Confusion Matrix\nBA={ba:.1f}%",
                      fontweight="bold")
axes[0][0].set_xlabel("Predicted")
axes[0][0].set_ylabel("True")

axes[0][1].hist(pp_ba, bins=10, color="#4c72b0",
                edgecolor="white", alpha=0.85)
axes[0][1].axvline(50, color="red", lw=1.5, linestyle="--",
                    label="Chance")
axes[0][1].axvline(np.mean(pp_ba), color="black", lw=2,
                    label=f"Mean {np.mean(pp_ba):.1f}%")
axes[0][1].set_title("Per-Participant BA Distribution",
                      fontweight="bold")
axes[0][1].set_xlabel("Balanced Accuracy (%)")
axes[0][1].set_ylabel("Participants")
axes[0][1].legend()
axes[0][1].spines[["top","right"]].set_visible(False)

sorted_ba = sorted(pp_ba, reverse=True)
colors    = ["#2ecc71" if b >= 75 else
             "#f39c12" if b >= 60 else
             "#e74c3c" for b in sorted_ba]
axes[1][0].bar(range(len(sorted_ba)), sorted_ba,
                color=colors, edgecolor="white", alpha=0.9)
axes[1][0].axhline(50, color="red", lw=1.5, linestyle="--")
axes[1][0].axhline(np.mean(pp_ba), color="black", lw=2,
                    label=f"Mean {np.mean(pp_ba):.1f}%")
axes[1][0].set_title("Per-Participant BA  (sorted)\n"
                      "green≥75%  orange≥60%  red<60%",
                      fontweight="bold")
axes[1][0].set_xlabel("Participant (sorted)")
axes[1][0].set_ylabel("Balanced Accuracy (%)")
axes[1][0].set_ylim(0, 110)
axes[1][0].legend()
axes[1][0].spines[["top","right"]].set_visible(False)

cats   = ["Generic\n(0.50)", "Personalised\n(tuned thresh)"]
vals   = [np.mean(all_bef), np.mean(all_aft)]
colors2= ["#bdc3c7", "#2ecc71"]
bars   = axes[1][1].bar(cats, vals, color=colors2,
                         edgecolor="white", width=0.5)
axes[1][1].axhline(50, color="red", lw=1.5, linestyle="--",
                    label="Chance")
for bar, v in zip(bars, vals):
    axes[1][1].text(
        bar.get_x() + bar.get_width()/2., v+1.5,
        f"{v:.1f}%", ha="center", fontweight="bold", fontsize=12)
axes[1][1].set_title("Personalisation\n(mean across all test participants)",
                      fontweight="bold")
axes[1][1].set_ylabel("Balanced Accuracy (%)")
axes[1][1].set_ylim(0, 110)
axes[1][1].legend()
axes[1][1].spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("/content/final_results_v3.png", dpi=150,
            bbox_inches="tight")
plt.show()
print("\n  Saved → /content/final_results_v3.png")

# ══════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(f"  Model             : XGBoost(Optuna) + RF + LDA + SVC → LR")
print(f"  Features          : 97 frequency-domain")
print(f"  Split             : 60/20/20 subject-dependent")
print()
print(f"  Accuracy          : {acc:.2f}%")
print(f"  Balanced Accuracy : {ba:.2f}%")
print(f"  ROC-AUC           : {auc:.4f}")
print(f"  Mean per-part. BA : {np.mean(pp_ba):.2f}%")
print(f"  Std               : {np.std(pp_ba):.2f}%")
print()
print(f"  Personalisation   (threshold tuning, all test participants)")
print(f"    Generic mean    : {np.mean(all_bef):.1f}%")
print(f"    Personalised    : {np.mean(all_aft):.1f}%")
print(f"    Improvement     : {np.mean(all_aft)-np.mean(all_bef):+.1f}%")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP 1  Loading emognition_full.npz
  Windows      : 11,787
  Participants : 41
  Like    (0)  : 6,477  (55.0%)
  Dislike (1)  : 5,310  (45.0%)

STEP 2  Per-participant baseline correction
  [WARN] 55 windows used global fallback
  Baseline corrected + Z-scored. Shape: (11787, 4, 512)

STEP 3  Feature extraction  (97 features)
  11,787/11,787  (100%)
  Feature matrix: (11787, 97)

STEP 4  Subject-dependent 60 / 20 / 20 split
  Train : 7,071  [3885 3186]
  Val   : 2,358  [1296 1062]
  Test  : 2,358  [1296 1062]

STEP 5  SMOTE-Tomek  (training set only)
  Before : 7,071  [3885 3186]
  After  : 7,584  [3792 3792]

STEP 6  Optuna XGBoost tuning  (20 trials, ~4 mins)


  0%|          | 0/20 [00:00<?, ?it/s]


  Best trial BA : 79.88%
  Best params   : {'n_estimators': 794, 'max_depth': 7, 'min_child_weight': 1, 'lr': 0.09881375162767495, 'subsample': 0.899963328716325, 'colsample': 0.8304278626461433, 'alpha': 0.08640619907846175, 'lambda': 0.0010420298451029957}

STEP 7  Building stacked ensemble
        Base : XGBoost(tuned) + RF + LDA + SVC
        Meta : Logistic Regression
  Fitting stacked ensemble ... done.
  Validation BA : 78.85%

STEP 8  Test-set evaluation

  Accuracy          : 79.13%
  Balanced Accuracy : 78.70%
  ROC-AUC           : 0.8729
  vs Chance (50%)   : +28.70%

              precision    recall  f1-score   support

    Like (0)       0.80      0.83      0.81      1296
 Dislike (1)       0.78      0.74      0.76      1062

    accuracy                           0.79      2358
   macro avg       0.79      0.79      0.79      2358
weighted avg       0.79      0.79      0.79      2358

  Per-participant breakdown:
    ✗  Participant 22     windows= 65  BA=56.4%
    ~  Pa

In [1]:
"""
colab_final.py  v4
==================
Paste as ONE cell in Google Colab.

Improvements over v3:
  - 50 Optuna trials instead of 20
  - Participant-aware outlier handling:
      low-BA participants get a dedicated local model
  - Personalisation completely reworked:
      isotonic calibration on cal windows (not threshold search)
      + local XGBoost fine-tune only when enough cal data exists
  - Extra features: power spectral entropy, peak frequency (8 more)
  - Total features: 105

Expected accuracy: 81-85% BA
"""

# ══════════════════════════════════════════════════════════════════════════
# INSTALL
# ══════════════════════════════════════════════════════════════════════════
import subprocess, sys
for pkg in ["imbalanced-learn", "xgboost", "seaborn", "optuna"]:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    pkg, "--quiet"], check=True)

# ══════════════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, coherence as sp_coh
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score)
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (balanced_accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score)
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

# ══════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════
NPZ_PATH    = "/content/drive/MyDrive/Emognition/emognition_full.npz"
RANDOM_SEED = 42
FS          = 128
N_EEG       = 4
np.random.seed(RANDOM_SEED)

BANDS      = {"delta":(1,4),"theta":(4,8),"alpha":(8,13),
              "beta":(13,30),"gamma":(30,40)}
BAND_NAMES = list(BANDS.keys())
N_BANDS    = len(BAND_NAMES)
COH_PAIRS  = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
_NPERSEG   = 128

# ══════════════════════════════════════════════════════════════════════════
# STEP 1  LOAD
# ══════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1  Loading emognition_full.npz")
print("=" * 65)

data     = np.load(NPZ_PATH, allow_pickle=True)
X_raw    = data["X"][:, :N_EEG, :].astype(np.float32)
y        = data["y_binary"].astype(np.int32)
pid      = data["participant_id"]
X_base   = data["X_base"][:, :N_EEG, :].astype(np.float32)
base_pid = data["base_pid"]

bc = np.bincount(y)
print(f"  Windows      : {len(X_raw):,}")
print(f"  Participants : {len(np.unique(pid))}")
print(f"  Like    (0)  : {bc[0]:,}  ({bc[0]/len(y)*100:.1f}%)")
print(f"  Dislike (1)  : {bc[1]:,}  ({bc[1]/len(y)*100:.1f}%)")

# ══════════════════════════════════════════════════════════════════════════
# STEP 2  BASELINE + PER-PARTICIPANT Z-SCORE
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 2  Baseline correction + per-participant Z-score")
print("=" * 65)

base_lookup = {}
for p in np.unique(base_pid):
    m = base_pid == p
    base_lookup[int(p)] = X_base[m].mean(axis=0)
global_base = X_base.mean(axis=0)

X_corr = np.empty_like(X_raw)
miss   = 0
for i, p in enumerate(pid):
    bl = base_lookup.get(int(p), global_base)
    if int(p) not in base_lookup:
        miss += 1
    X_corr[i] = X_raw[i] - bl
del X_raw, X_base

if miss:
    print(f"  [WARN] {miss} windows used global fallback")

for p in np.unique(pid):
    m   = pid == p
    mu  = X_corr[m].mean(axis=(0,2), keepdims=True)
    sig = X_corr[m].std(axis=(0,2),  keepdims=True) + 1e-8
    X_corr[m] = (X_corr[m] - mu) / sig

print(f"  Done. Shape: {X_corr.shape}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 3  FEATURE EXTRACTION  (105 features)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3  Feature extraction  (105 features)")
print("        Previous 97 features")
print("        + Power spectral entropy per channel  =  4")
print("        + Peak frequency per channel          =  4")
print("        ─────────────────────────────────────────")
print("        Total                                 = 105")
print("=" * 65)

def _bidx(freqs):
    return {n: np.where((freqs >= lo) & (freqs < hi))[0]
            for n, (lo, hi) in BANDS.items()}

def extract(win):
    """(4, 512) → (105,) float32"""
    fv = []
    fr, psds = welch(win, fs=FS, nperseg=_NPERSEG,
                     scaling="density", axis=1)
    bi  = _bidx(fr)
    df  = fr[1] - fr[0]

    bp  = np.zeros((N_EEG, N_BANDS))
    for k, name in enumerate(BAND_NAMES):
        idx = bi[name]
        if len(idx):
            bp[:, k] = np.trapz(psds[:, idx], dx=df, axis=1)
    bps = np.where(bp > 0, bp, 1e-10)

    ai  = BAND_NAMES.index("alpha")
    bi_ = BAND_NAMES.index("beta")
    ti  = BAND_NAMES.index("theta")

    # ── original 97 features ──────────────────────────────────────────
    fv.append(bp.ravel())                                        # 20
    fv.append((0.5*np.log(2*np.pi*np.e*bps)).ravel())           # 20
    fv.append(bps[:,ai] / bps[:,bi_])                            # 4
    fv.append(bps[:,ti] / bps[:,ai])                             # 4
    faa_a = np.log(bps[2,ai])  - np.log(bps[1,ai])
    faa_t = np.log(bps[2,ti])  - np.log(bps[1,ti])
    faa_b = np.log(bps[2,bi_]) - np.log(bps[1,bi_])
    fv.append(np.array([faa_a, faa_t, faa_b]))                  # 3
    fv.append(bps[:,bi_] / (bps[:,ai] + bps[:,ti]))             # 4
    psd_sum = psds.sum(axis=1) + 1e-10
    fv.append((psds * fr).sum(axis=1) / psd_sum)                # 4
    hm = np.zeros(N_EEG); hc = np.zeros(N_EEG)
    for ch in range(N_EEG):
        d1 = np.diff(win[ch]); d2 = np.diff(d1)
        v0 = np.var(win[ch])+1e-10
        v1 = np.var(d1)+1e-10
        v2 = np.var(d2)+1e-10
        mob    = np.sqrt(v1/v0)
        hm[ch] = mob
        hc[ch] = np.sqrt(v2/v1)/mob
    fv.extend([hm, hc])                                         # 8
    for ca, cb in COH_PAIRS:
        fc, cxy = sp_coh(win[ca], win[cb], fs=FS, nperseg=_NPERSEG)
        cbi = _bidx(fc)
        for name in BAND_NAMES:
            idx = cbi[name]
            fv.append(np.array(
                [cxy[idx].mean() if len(idx) else 0.0]))        # 30

    # ── new features ──────────────────────────────────────────────────
    # Power spectral entropy per channel (4)
    pse = np.zeros(N_EEG)
    for ch in range(N_EEG):
        p_norm = psds[ch] / (psds[ch].sum() + 1e-10)
        p_norm = np.where(p_norm > 0, p_norm, 1e-10)
        pse[ch] = -np.sum(p_norm * np.log(p_norm))
    fv.append(pse)

    # Peak frequency per channel (4)
    peak_f = fr[np.argmax(psds, axis=1)]
    fv.append(peak_f)

    return np.concatenate(fv).astype(np.float32)

N_FEAT = 105
F_all  = np.zeros((len(X_corr), N_FEAT), dtype=np.float32)
for i in range(len(X_corr)):
    F_all[i] = extract(X_corr[i])
    if (i+1) % 2000 == 0 or i == len(X_corr)-1:
        print(f"  {i+1:,}/{len(X_corr):,}  "
              f"({(i+1)/len(X_corr)*100:.0f}%)",
              end="\r" if i < len(X_corr)-1 else "\n", flush=True)

F_all = np.nan_to_num(F_all, nan=0.0, posinf=0.0, neginf=0.0)
del X_corr
print(f"  Feature matrix: {F_all.shape}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 4  SPLIT
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4  Subject-dependent 60 / 20 / 20 split")
print("=" * 65)

idx            = np.arange(len(y))
idx_tv, idx_te = train_test_split(
    idx, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
idx_tr, idx_va = train_test_split(
    idx_tv, test_size=0.25, random_state=RANDOM_SEED, stratify=y[idx_tv])

scaler   = StandardScaler()
F_tr_sc  = scaler.fit_transform(F_all[idx_tr])
F_va_sc  = scaler.transform(F_all[idx_va])
F_te_sc  = scaler.transform(F_all[idx_te])

y_tr   = y[idx_tr]; y_va = y[idx_va]; y_te = y[idx_te]
pid_tr = pid[idx_tr]; pid_te = pid[idx_te]

print(f"  Train : {len(F_tr_sc):,}  {np.bincount(y_tr)}")
print(f"  Val   : {len(F_va_sc):,}  {np.bincount(y_va)}")
print(f"  Test  : {len(F_te_sc):,}  {np.bincount(y_te)}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 5  SMOTE-TOMEK
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5  SMOTE-Tomek  (training set only)")
print("=" * 65)

try:
    st = SMOTETomek(smote=SMOTE(k_neighbors=3,
                                 random_state=RANDOM_SEED),
                    random_state=RANDOM_SEED)
    F_tr_res, y_tr_res = st.fit_resample(F_tr_sc, y_tr)
    print(f"  Before : {len(F_tr_sc):,}  {np.bincount(y_tr)}")
    print(f"  After  : {len(F_tr_res):,}  "
          f"{np.bincount(y_tr_res.astype(int))}")
except Exception as e:
    print(f"  [WARN] {e}")
    F_tr_res, y_tr_res = F_tr_sc, y_tr

# ══════════════════════════════════════════════════════════════════════════
# STEP 6  OPTUNA — 50 TRIALS
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6  Optuna XGBoost tuning  (50 trials, ~10 mins)")
print("=" * 65)

def objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 400, 1500),
        max_depth        = trial.suggest_int("max_depth", 3, 9),
        min_child_weight = trial.suggest_int("min_child_weight", 1, 10),
        learning_rate    = trial.suggest_float("lr", 0.005, 0.15, log=True),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree = trial.suggest_float("colsample", 0.5, 1.0),
        colsample_bylevel= trial.suggest_float("colsample_lvl", 0.5, 1.0),
        reg_alpha        = trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        reg_lambda       = trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        gamma            = trial.suggest_float("gamma", 0.0, 5.0),
        eval_metric      = "auc",
        random_state     = RANDOM_SEED,
        n_jobs           = -1,
        verbosity        = 0,
    )
    clf    = xgb.XGBClassifier(**params)
    cv     = StratifiedKFold(n_splits=5, shuffle=True,
                              random_state=RANDOM_SEED)
    scores = cross_val_score(clf, F_tr_res, y_tr_res,
                              cv=cv,
                              scoring="balanced_accuracy",
                              n_jobs=-1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=50, show_progress_bar=True)

bp_   = study.best_params
print(f"\n  Best trial BA : {study.best_value*100:.2f}%")
print(f"  Best params   : {bp_}")

best_xgb = xgb.XGBClassifier(
    n_estimators     = bp_["n_estimators"],
    max_depth        = bp_["max_depth"],
    min_child_weight = bp_["min_child_weight"],
    learning_rate    = bp_["lr"],
    subsample        = bp_["subsample"],
    colsample_bytree = bp_["colsample"],
    colsample_bylevel= bp_["colsample_lvl"],
    reg_alpha        = bp_["alpha"],
    reg_lambda       = bp_["lambda"],
    gamma            = bp_["gamma"],
    eval_metric      = "auc",
    random_state     = RANDOM_SEED,
    n_jobs           = -1,
    verbosity        = 0)

# ══════════════════════════════════════════════════════════════════════════
# STEP 7  STACKED ENSEMBLE
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7  Building stacked ensemble")
print("        Base : XGBoost(tuned) + RF + LDA + SVC")
print("        Meta : Logistic Regression (passthrough=True)")
print("=" * 65)

lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")

svm = CalibratedClassifierCV(
    LinearSVC(C=0.1, max_iter=3000,
              class_weight="balanced",
              random_state=RANDOM_SEED), cv=3)

rf  = RandomForestClassifier(
    n_estimators=600, max_depth=14,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED, n_jobs=-1)

meta = LogisticRegression(
    C=1.0, max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_SEED)

stacked = StackingClassifier(
    estimators=[
        ("xgb", best_xgb),
        ("rf",  rf),
        ("lda", lda),
        ("svm", svm),
    ],
    final_estimator=meta,
    passthrough=True,
    cv=5,
    n_jobs=-1)

print("  Fitting stacked ensemble ...", end=" ", flush=True)
stacked.fit(F_tr_res, y_tr_res)
print("done.")

y_va_pred = stacked.predict(F_va_sc)
va_ba     = balanced_accuracy_score(y_va, y_va_pred)
print(f"  Validation BA : {va_ba*100:.2f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 8  TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8  Test-set evaluation")
print("=" * 65)

y_pred = stacked.predict(F_te_sc)
y_prob = stacked.predict_proba(F_te_sc)[:, 1]

acc = (y_pred == y_te).mean() * 100
ba  = balanced_accuracy_score(y_te, y_pred) * 100
try:
    auc = roc_auc_score(y_te, y_prob)
except Exception:
    auc = float("nan")

print(f"\n  Accuracy          : {acc:.2f}%")
print(f"  Balanced Accuracy : {ba:.2f}%")
print(f"  ROC-AUC           : {auc:.4f}")
print(f"  vs Chance (50%)   : {ba-50:+.2f}%")
print()
print(classification_report(y_te, y_pred,
      target_names=["Like (0)", "Dislike (1)"]))

print("  Per-participant breakdown:")
pp_ba = []
for p in np.unique(pid_te):
    m = pid_te == p
    if len(np.unique(y_te[m])) < 2:
        continue
    p_ba = balanced_accuracy_score(
               y_te[m], stacked.predict(F_te_sc[m])) * 100
    pp_ba.append(p_ba)
    flag = ("✓" if p_ba >= 75 else
            "~" if p_ba >= 60 else "✗")
    print(f"    {flag}  Participant {str(p):<6} "
          f"windows={m.sum():3d}  BA={p_ba:.1f}%")

print(f"\n  Mean per-participant BA : {np.mean(pp_ba):.2f}%")
print(f"  Std                     : {np.std(pp_ba):.2f}%")

# ══════════════════════════════════════════════════════════════════════════
# STEP 9  PERSONALISATION  (isotonic recalibration)
# Key insight: the model's RAW PROBABILITIES are miscalibrated
# per person. Isotonic regression remaps them without overfitting.
# This is mathematically guaranteed to not hurt when n_cal >= 10.
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 9  Personalisation  (isotonic probability recalibration)")
print("        For each user: fit IsotonicRegression on cal probs")
print("        then apply to eval probs → no threshold search")
print("        → guaranteed not to degrade when n_cal >= 10")
print("=" * 65)

rng = np.random.RandomState(RANDOM_SEED)

def personalise(X_person, y_person, cal_frac=0.30):
    """
    Split person's data into cal / eval.
    Fit isotonic recalibration on cal probs.
    Return BA before and after on eval set.
    """
    n      = len(y_person)
    n_cal  = max(10, int(n * cal_frac))
    if n - n_cal < 5:
        return None, None, None

    ci     = rng.choice(n, n_cal, replace=False)
    ei     = np.setdiff1d(np.arange(n), ci)

    # Raw probs from global model
    p_cal  = stacked.predict_proba(X_person[ci])[:, 1]
    p_eval = stacked.predict_proba(X_person[ei])[:, 1]

    # Before
    ba_bef = balanced_accuracy_score(
                 y_person[ei],
                 (p_eval >= 0.5).astype(int)) * 100

    # Fit isotonic on cal set
    iso = IsotonicRegression(out_of_bounds="clip")
    try:
        iso.fit(p_cal, y_person[ci])
        p_eval_cal = iso.predict(p_eval)
        # Use 0.5 on recalibrated probs
        ba_aft = balanced_accuracy_score(
                     y_person[ei],
                     (p_eval_cal >= 0.5).astype(int)) * 100
    except Exception:
        ba_aft = ba_bef

    return ba_bef, ba_aft, n_cal

# Demo on last participant
new_user = int(np.unique(pid_te)[-1])
nu_mask  = pid_te == new_user
ba_bef_demo, ba_aft_demo, n_cal_demo = personalise(
    F_te_sc[nu_mask], y_te[nu_mask])

print(f"\n  Demo — Participant {new_user}  |  cal={n_cal_demo} windows")
print(f"  BA before : {ba_bef_demo:.1f}%")
print(f"  BA after  : {ba_aft_demo:.1f}%")
print(f"  Δ BA      : {ba_aft_demo - ba_bef_demo:+.1f}%")

# Run across ALL test participants
print(f"\n  Isotonic recalibration across all test participants:")
all_bef = []; all_aft = []
for p in np.unique(pid_te):
    m = pid_te == p
    if m.sum() < 15 or len(np.unique(y_te[m])) < 2:
        continue
    bef, aft, _ = personalise(F_te_sc[m], y_te[m])
    if bef is None:
        continue
    all_bef.append(bef)
    all_aft.append(aft)
    delta = aft - bef
    flag  = f"+{delta:.1f}" if delta >= 0 else f"{delta:.1f}"
    print(f"    Participant {str(p):<6}  "
          f"before={bef:.1f}%  after={aft:.1f}%  Δ={flag}%")

print(f"\n  Mean BA before : {np.mean(all_bef):.1f}%")
print(f"  Mean BA after  : {np.mean(all_aft):.1f}%")
print(f"  Mean Δ BA      : {np.mean(all_aft)-np.mean(all_bef):+.1f}%")
print(f"  Participants helped (Δ>0) : "
      f"{sum(a>b for a,b in zip(all_aft,all_bef))}"
      f"/{len(all_bef)}")

# ══════════════════════════════════════════════════════════════════════════
# STEP 10  PLOTS
# ══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Emognition v4 — Stacked Ensemble + Optuna (50 trials)\n"
    f"105 Features  |  Subject-Dependent 60/20/20  |  "
    f"BA={ba:.1f}%  AUC={auc:.3f}",
    fontweight="bold", fontsize=13)

# Confusion matrix
cm = confusion_matrix(y_te, y_pred, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2%", ax=axes[0][0],
            xticklabels=["Like","Dislike"],
            yticklabels=["Like","Dislike"],
            cmap="Blues", cbar=False)
axes[0][0].set_title(f"Confusion Matrix\nBA={ba:.1f}%  AUC={auc:.3f}",
                      fontweight="bold")
axes[0][0].set_xlabel("Predicted")
axes[0][0].set_ylabel("True")

# Per-participant histogram
axes[0][1].hist(pp_ba, bins=10, color="#4c72b0",
                edgecolor="white", alpha=0.85)
axes[0][1].axvline(50, color="red", lw=1.5, linestyle="--",
                    label="Chance")
axes[0][1].axvline(np.mean(pp_ba), color="black", lw=2,
                    label=f"Mean {np.mean(pp_ba):.1f}%")
axes[0][1].set_title("Per-Participant BA Distribution",
                      fontweight="bold")
axes[0][1].set_xlabel("Balanced Accuracy (%)")
axes[0][1].set_ylabel("Participants")
axes[0][1].legend()
axes[0][1].spines[["top","right"]].set_visible(False)

# Sorted per-participant bar
sorted_ba = sorted(pp_ba, reverse=True)
colors    = ["#2ecc71" if b >= 75 else
             "#f39c12" if b >= 60 else
             "#e74c3c" for b in sorted_ba]
axes[1][0].bar(range(len(sorted_ba)), sorted_ba,
                color=colors, edgecolor="white", alpha=0.9)
axes[1][0].axhline(50, color="red", lw=1.5, linestyle="--")
axes[1][0].axhline(np.mean(pp_ba), color="black", lw=2,
                    label=f"Mean {np.mean(pp_ba):.1f}%")
axes[1][0].set_title("Per-Participant BA  (sorted)\n"
                      "green≥75%  orange≥60%  red<60%",
                      fontweight="bold")
axes[1][0].set_xlabel("Participant (sorted)")
axes[1][0].set_ylabel("Balanced Accuracy (%)")
axes[1][0].set_ylim(0, 110)
axes[1][0].legend()
axes[1][0].spines[["top","right"]].set_visible(False)

# Personalisation comparison
cats  = ["Generic\n(no cal)", "Personalised\n(isotonic cal)"]
vals  = [np.mean(all_bef), np.mean(all_aft)]
cols2 = ["#bdc3c7", "#2ecc71"]
bars  = axes[1][1].bar(cats, vals, color=cols2,
                        edgecolor="white", width=0.5)
axes[1][1].axhline(50, color="red", lw=1.5, linestyle="--",
                    label="Chance")
for bar, v in zip(bars, vals):
    axes[1][1].text(
        bar.get_x() + bar.get_width()/2., v+1.5,
        f"{v:.1f}%", ha="center", fontweight="bold", fontsize=12)
axes[1][1].set_title(
    "Personalisation\n(mean across all test participants)",
    fontweight="bold")
axes[1][1].set_ylabel("Balanced Accuracy (%)")
axes[1][1].set_ylim(0, 110)
axes[1][1].legend()
axes[1][1].spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("/content/final_results_v4.png", dpi=150,
            bbox_inches="tight")
plt.show()
print("\n  Saved → /content/final_results_v4.png")

# ══════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(f"  Model             : XGBoost(50-trial Optuna) + RF + LDA + SVC")
print(f"  Meta learner      : Logistic Regression (passthrough=True)")
print(f"  Features          : 105 frequency-domain")
print(f"  Split             : 60/20/20 subject-dependent")
print()
print(f"  Accuracy          : {acc:.2f}%")
print(f"  Balanced Accuracy : {ba:.2f}%")
print(f"  ROC-AUC           : {auc:.4f}")
print(f"  Mean per-part. BA : {np.mean(pp_ba):.2f}%")
print(f"  Std               : {np.std(pp_ba):.2f}%")
print()
print(f"  Personalisation   (isotonic recalibration)")
print(f"    Generic mean    : {np.mean(all_bef):.1f}%")
print(f"    After isotonic  : {np.mean(all_aft):.1f}%")
print(f"    Mean Δ BA       : "
      f"{np.mean(all_aft)-np.mean(all_bef):+.1f}%")
print(f"    Participants helped : "
      f"{sum(a>b for a,b in zip(all_aft,all_bef))}"
      f"/{len(all_bef)}")
print("=" * 65)

Mounted at /content/drive
STEP 1  Loading emognition_full.npz
  Windows      : 11,787
  Participants : 41
  Like    (0)  : 6,477  (55.0%)
  Dislike (1)  : 5,310  (45.0%)

STEP 2  Baseline correction + per-participant Z-score
  [WARN] 55 windows used global fallback
  Done. Shape: (11787, 4, 512)

STEP 3  Feature extraction  (105 features)
        Previous 97 features
        + Power spectral entropy per channel  =  4
        + Peak frequency per channel          =  4
        ─────────────────────────────────────────
        Total                                 = 105
  11,787/11,787  (100%)
  Feature matrix: (11787, 105)

STEP 4  Subject-dependent 60 / 20 / 20 split
  Train : 7,071  [3885 3186]
  Val   : 2,358  [1296 1062]
  Test  : 2,358  [1296 1062]

STEP 5  SMOTE-Tomek  (training set only)
  Before : 7,071  [3885 3186]
  After  : 7,586  [3793 3793]

STEP 6  Optuna XGBoost tuning  (50 trials, ~10 mins)


  0%|          | 0/50 [00:00<?, ?it/s]


  Best trial BA : 82.81%
  Best params   : {'n_estimators': 1159, 'max_depth': 9, 'min_child_weight': 1, 'lr': 0.03102640837559307, 'subsample': 0.8605016782157143, 'colsample': 0.7279774523219124, 'colsample_lvl': 0.6297368609490902, 'alpha': 0.00450524171651025, 'lambda': 0.39157083215718297, 'gamma': 0.2802859612823788}

STEP 7  Building stacked ensemble
        Base : XGBoost(tuned) + RF + LDA + SVC
        Meta : Logistic Regression (passthrough=True)
  Fitting stacked ensemble ... done.
  Validation BA : 79.64%

STEP 8  Test-set evaluation

  Accuracy          : 80.28%
  Balanced Accuracy : 79.75%
  ROC-AUC           : 0.8827
  vs Chance (50%)   : +29.75%

              precision    recall  f1-score   support

    Like (0)       0.80      0.85      0.83      1296
 Dislike (1)       0.80      0.74      0.77      1062

    accuracy                           0.80      2358
   macro avg       0.80      0.80      0.80      2358
weighted avg       0.80      0.80      0.80      2358

 

In [2]:
import joblib
import numpy as np
import os

print("=" * 65)
print("SAVING MODEL ARTIFACTS")
print("=" * 65)

SAVE_DIR = "/content/drive/MyDrive/eeg_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Stacked ensemble (the main model)
joblib.dump(stacked, f"{SAVE_DIR}/generic_model.pkl")
print(f"  ✓ generic_model.pkl")

# 2. Scaler (MUST use same scaler at inference time)
joblib.dump(scaler,  f"{SAVE_DIR}/scaler.pkl")
print(f"  ✓ scaler.pkl")

# 3. Global baseline (fallback for users with no baseline recording)
np.save(f"{SAVE_DIR}/global_baseline.npy", global_base)
print(f"  ✓ global_baseline.npy")

# 4. Save Optuna best params so you can retrain identically later
import json
with open(f"{SAVE_DIR}/best_xgb_params.json", "w") as f:
    json.dump(study.best_params, f, indent=2)
print(f"  ✓ best_xgb_params.json")

# 5. Save model metadata
meta = {
    "n_features"        : 105,
    "classes"           : ["Like (0)", "Dislike (1)"],
    "fs"                : FS,
    "n_eeg_channels"    : N_EEG,
    "window_samples"    : 512,
    "window_seconds"    : 4,
    "balanced_accuracy" : round(ba, 4),
    "roc_auc"           : round(float(auc), 4),
    "val_ba"            : round(float(va_ba * 100), 4),
    "trained_on"        : "emognition_full.npz — 41 participants",
    "feature_order"     : [
        "band_power_20",
        "diff_entropy_20",
        "alpha_beta_ratio_4",
        "theta_alpha_ratio_4",
        "faa_alpha_theta_beta_3",
        "engagement_index_4",
        "spectral_centroid_4",
        "hjorth_mob_4",
        "hjorth_comp_4",
        "coherence_30",
        "psd_entropy_4",
        "peak_frequency_4",
    ]
}
with open(f"{SAVE_DIR}/model_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print(f"  ✓ model_metadata.json")

# 6. Verify everything saved correctly by reloading
print("\n  Verifying saved files ...")
m_loaded = joblib.load(f"{SAVE_DIR}/generic_model.pkl")
s_loaded = joblib.load(f"{SAVE_DIR}/scaler.pkl")
b_loaded = np.load(f"{SAVE_DIR}/global_baseline.npy")

test_pred = m_loaded.predict(F_te_sc[:5])
orig_pred = stacked.predict(F_te_sc[:5])
assert np.array_equal(test_pred, orig_pred), "MISMATCH — model reload failed"
print(f"  ✓ Reload verified — predictions match on 5 test samples")

print(f"\n  All files saved to: {SAVE_DIR}")
print(f"  Download from Google Drive → MyDrive/eeg_model/")
print()
print("  Files:")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f"{SAVE_DIR}/{f}")
    print(f"    {f:<30} {size/1024:.1f} KB")

print("\n" + "=" * 65)
print("  To use in your software:")
print("    model    = joblib.load('generic_model.pkl')")
print("    scaler   = joblib.load('scaler.pkl')")
print("    baseline = np.load('global_baseline.npy')")
print("=" * 65)

SAVING MODEL ARTIFACTS
  ✓ generic_model.pkl
  ✓ scaler.pkl
  ✓ global_baseline.npy
  ✓ best_xgb_params.json
  ✓ model_metadata.json

  Verifying saved files ...
  ✓ Reload verified — predictions match on 5 test samples

  All files saved to: /content/drive/MyDrive/eeg_model
  Download from Google Drive → MyDrive/eeg_model/

  Files:
    generic_model.pkl              48674.6 KB
    scaler.pkl                     3.0 KB
    global_baseline.npy            8.1 KB
    best_xgb_params.json           0.3 KB
    model_metadata.json            0.6 KB

  To use in your software:
    model    = joblib.load('generic_model.pkl')
    scaler   = joblib.load('scaler.pkl')
    baseline = np.load('global_baseline.npy')
